# Prithvi — Prétraitement, inférence et extraction de features

Ce notebook montre un flux complet minimal : création d'une petite image TIF de test, prétraitement, tentative de chargement du modèle `PrithviClient` depuis `geocongoai` (via `terratorch`) et extraction de features.
**Remarques** : installez les extras `geocongoai[ia]` si vous voulez exécuter l'inférence localement (torch + terratorch).

In [ ]:
# Vérifier environnement et imports
import sys
print('Python', sys.version)
try:
    import rasterio, numpy as np, matplotlib.pyplot as plt
    from geocongoai.ia.prithvi import PrithviClient
    print('Imports OK: rasterio, numpy, geocongoai')
except Exception as e:
    print('Imports manquants ou warning:', e)
    # Si vous exécutez localement, installez: pip install geocongoai[ia]

In [ ]:
# Créer un petit TIF de test (3 bandes) pour démonstration
import numpy as np
import rasterio
from rasterio.transform import from_origin
height, width = 64, 64
bands = 3
arr = (np.random.rand(bands, height, width) * 1000).astype('uint16')
transform = from_origin(0, 0, 10, 10)
meta = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': bands,
    'dtype': 'uint16',
    'transform': transform
}
tif_path = 'examples/data/sample_ms.tif'
import os
os.makedirs(os.path.dirname(tif_path), exist_ok=True)
with rasterio.open(tif_path, 'w', **meta) as dst:
    dst.write(arr)
print('Wrote sample TIF to', tif_path)

In [ ]:
# Prétraitement minimal : normalisation simple par bande et visualisation
import rasterio, numpy as np, matplotlib.pyplot as plt
with rasterio.open(tif_path) as src:
    ms = src.read().astype(np.float32)
# Normaliser chaque bande entre 0 et 1
ms_norm = np.empty_like(ms, dtype=np.float32)
for i in range(ms.shape[0]):
    b = ms[i]
    mn, mx = b.min(), b.max()
    if mx - mn == 0:
        ms_norm[i] = 0
    else:
        ms_norm[i] = (b - mn) / (mx - mn)
# Afficher la composition RGB (bandes 1,2,3)
rgb = np.dstack([ms_norm[0], ms_norm[1], ms_norm[2]])
plt.figure(figsize=(4,4)); plt.imshow(rgb); plt.title('Sample RGB'); plt.axis('off')

In [ ]:
# Inférence: tenter de charger PrithviClient et d'extraire des features
from geocongoai.ia.prithvi import PrithviClient
client = PrithviClient(model_name='prithvi_eo_v2_300', pretrained=True)
try:
    client.load_model()
    print('Model loaded')
except Exception as e:
    print('Model load failed (expected if terratorch/torch absent):', e)
    # Nous continuons en mode démo sans inference réelle
try:
    res = client.extract_deep_features(tif_path)
    print('Extracted features keys:', list(res.keys()))
    # Afficher forme si possible
    feats = res.get('features')
    print('Features type:', type(feats))
except Exception as e:
    print('extract_deep_features failed or returned placeholder:', e)
    res = {'note': 'inference not executed; see notebook for setup steps'}
res

## Conclusion
Ce notebook fournit un canevas reproductible pour prétraiter des tuiles et exécuter le client `PrithviClient`. Pour l'utiliser en production, installez `torch` et `terratorch`, utilisez des tuiles plus grandes et géoréférez correctement les tifs.